# Wave 01 - step 1: extract and load

Reads landed files from S3 and bulk-loads them into a Snowflake staging table. Runs as task `extract_load` of every wave_01 job; `02_transform` runs after it. Parameters come from `resources/jobs/wave_01_*.job.yml`.

In [0]:
import os

# Defaults are for interactive runs; job parameters override them.
_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
_default_src = "/Workspace" + _nb_path.split("/notebooks/")[0] + "/src"

dbutils.widgets.text("job_name", "wave_01_interactive")
dbutils.widgets.text("src_path", _default_src)
dbutils.widgets.text("catalog", "dev_catalog")
dbutils.widgets.text("schema", "talend_migration")
dbutils.widgets.text("snowflake_secret_scope", "snowflake-dev")
dbutils.widgets.text("snowflake_schema", "")  # required: your dev shadow schema, e.g. MIGRATION_DEV_<short_name>_main
dbutils.widgets.text("source_path", "")
dbutils.widgets.text("source_format", "csv")
dbutils.widgets.text("snowflake_stage_table", "")

p = {k: dbutils.widgets.get(k) for k in [
    "job_name", "src_path", "catalog", "schema", "snowflake_secret_scope", "snowflake_schema",
    "source_path", "source_format", "snowflake_stage_table",
]}
print(p)

In [0]:
import sys

if p["src_path"] not in sys.path:
    sys.path.insert(0, p["src_path"])

from operators import SnowflakeConfig, read_files, write_snowflake

if not p["snowflake_schema"]:
    raise ValueError("Set the snowflake_schema widget to your dev shadow schema (e.g. MIGRATION_DEV_<short_name>_main)")
# Unqualified Snowflake table names resolve in this schema.
sf_cfg = SnowflakeConfig.from_secrets(dbutils, scope=p["snowflake_secret_scope"], schema=p["snowflake_schema"])

In [0]:
df = read_files(spark, p["source_path"], fmt=p["source_format"])
row_count = df.count()
print(f"Read {row_count} rows from {p['source_path']}")

In [0]:
write_snowflake(df, sf_cfg, table=p["snowflake_stage_table"], mode="overwrite")
print(f"Loaded {row_count} rows into {p['snowflake_stage_table']}")

In [0]:
import json

# Downstream tasks can read this with dbutils.jobs.taskValues.get("extract_load", "rows_loaded").
dbutils.jobs.taskValues.set("rows_loaded", row_count)
dbutils.notebook.exit(json.dumps({"job_name": p["job_name"], "rows_loaded": row_count}))